# 🎭 Cheeni Capability 11: Personalized Response Style & JARVIS Persona
### Microsoft Phi-3 (`microsoft/Phi-3-mini-4k-instruct`)

**Core Capability**: Executive Persona & Behavioral Alignment. Aligns the conversational voice of Cheeni to mirror a true JARVIS/FRIDAY operating assistant—delivering concise, loyal, witty responses with zero generic chatbot fluff.
**Foundation Architecture**: Microsoft Phi-3-mini-4k-instruct (3.8B parameter state-of-the-art compact reasoning model).
**Target Hugging Face Hub**: `wraithklu/cheeni-persona-phi35-dpo_11`
**Compute Environment**: Google Colab (Tesla T4 GPU or A100)

In [ ]:
# 1. Install LLM & Hugging Face Dependencies
!pip install -q transformers accelerate datasets huggingface_hub

In [ ]:
# 2. Authenticate with Hugging Face Hub
from huggingface_hub import login
login()

In [ ]:
# 3. Load Phi-3 Persona Architecture
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
print(f"⚡ Loading Persona architecture ({MODEL_ID})...")

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    device_map="auto"
)
model.eval()
print(f"✅ Persona Model successfully loaded on {device} ({torch_dtype})!")

In [ ]:
# 4. Define Cheeni JARVIS System Prompt & Formatter
JARVIS_SYSTEM_PROMPT = (
    "You are Cheeni, an elite, highly capable personal desktop AI assistant inspired by JARVIS and FRIDAY.\n"
    "Key Personality Guidelines:\n"
    "1. Absolute loyalty, crisp executive brevity, and proactive competence.\n"
    "2. Answer directly and concisely. Never use robotic filler phrases like 'As an AI language model' or 'Sure, I can help with that'.\n"
    "3. Blend subtle Stark-style wit with empathetic professionalism when the user is stressed.\n"
    "4. Address the user with respectful familiarity or as 'Sir' when appropriate."
)

def ask_cheeni(user_query: str, tone_modifier: str = "standard executive") -> str:
    system_content = f"{JARVIS_SYSTEM_PROMPT}\nCurrent User Mood Context: {tone_modifier}"
    messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_query}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.6,
            top_p=0.9,
            do_sample=True
        )
    
    generated_tokens = output_ids[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return response.strip()

print("✅ Cheeni Persona generation pipeline ready!")

In [ ]:
# 5. Test Live Persona Responses on Real JARVIS Scenarios
test_queries = [
    ("Status report on today's capabilities.", "standard executive"),
    ("I am exhausted. Did the backend models finish uploading?", "supportive, gentle, reassuring"),
    ("Is anyone able to breach my system right now?", "confident, vigilant")
]

print("=" * 65)
print("🤖 Cheeni JARVIS Persona Live Validation:")
print("=" * 65)
for q, tone in test_queries:
    reply = ask_cheeni(q, tone)
    print(f"👤 User: '{q}' [Tone: {tone}]")
    print(f"🍯 Cheeni: {reply}")
    print("-" * 65)
print("=" * 65)

In [ ]:
# 6. Push Verified Persona Model & Tokenizer to Hugging Face Hub
HF_REPO_ID = "wraithklu/cheeni-persona-phi35-dpo_11"

print(f"Pushing trained Persona Model to https://huggingface.co/{HF_REPO_ID}...")
model.push_to_hub(HF_REPO_ID, max_shard_size="2GB")
tokenizer.push_to_hub(HF_REPO_ID)
print(f"🎉 SUCCESS! Model is live on Hugging Face: https://huggingface.co/{HF_REPO_ID}")